# Logistic Regression Baseline

In [1]:
from pathlib import Path
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

In [2]:
DATA_DIR = Path("data/merged")
GENRE = "MIXED"

train_df = pd.read_csv(DATA_DIR / f"{GENRE}_train.csv")
val_df = pd.read_csv(DATA_DIR / f"{GENRE}_val.csv")
test_df = pd.read_csv(DATA_DIR / f"{GENRE}_test.csv")

In [3]:
# define a logistic regression model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
    )),
    ("classifier", LogisticRegression(
        C=1.2,
        max_iter=1000,
    ))
])

# Train without automatically rendering the fitted pipeline in Jupyter.
_ = model.fit(
    train_df["review"],
    train_df["voted_up"]
)

In [4]:
# evaluate model performance
def evaluate_model(model, data):
    y_true = data["voted_up"]
    y_pred = model.predict(data["review"])

    print(classification_report(
        y_true, y_pred, labels=[1,0], target_names=["recommended", "not recommend"],
        zero_division=0, digits=3
    ))

    print("Macro-F1:", f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0,
    ))
    
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred, labels=[1,0]))

In [5]:
evaluate_model(model, val_df)

               precision    recall  f1-score   support

  recommended      0.881     0.804     0.841       876
not recommend      0.861     0.918     0.888      1158

     accuracy                          0.869      2034
    macro avg      0.871     0.861     0.865      2034
 weighted avg      0.870     0.869     0.868      2034

Macro-F1: 0.864510793296368
Confusion matrix:
[[ 704  172]
 [  95 1063]]


In [6]:
print("Test set results:")
evaluate_model(model, test_df)

Test set results:
               precision    recall  f1-score   support

  recommended      0.881     0.799     0.838       877
not recommend      0.858     0.918     0.887      1160

     accuracy                          0.867      2037
    macro avg      0.869     0.859     0.863      2037
 weighted avg      0.868     0.867     0.866      2037

Macro-F1: 0.8625729516467162
Confusion matrix:
[[ 701  176]
 [  95 1065]]


## External tests

Run the cells below in this notebook's trained kernel. They evaluate all games in
`config.EXTERNAL_TEST_GAME_IDS` and the balanced 4,000-review IMDb movie test set.
Reuse the current trained model and its training-time text processing; no fitting
or vocabulary construction is performed here. IMDb uses sentiment labels
(0=negative, 1=positive), while Steam uses recommendation labels.


In [7]:
import importlib
import config

# Refresh dataset configuration without restarting the trained kernel.
EXTERNAL_TEST_GAME_IDS = importlib.reload(config).EXTERNAL_TEST_GAME_IDS

external_test_dfs = {}
for game, appid in EXTERNAL_TEST_GAME_IDS.items():
    external_df = pd.read_csv(
        Path("data/processed") / f"reviews_{game}_{appid}.csv",
        usecols=["review", "voted_up"],
        dtype={"review": "string", "voted_up": "boolean"},
    )
    external_df = external_df.dropna(subset=["review", "voted_up"]).copy()
    external_df["voted_up"] = external_df["voted_up"].astype("int64")
    external_test_dfs[game] = external_df
    print(f"{game}: {len(external_df)} external test reviews")

# IMDb sentiment labels already use 0=negative and 1=positive.
imdb_test_df = pd.read_csv(
    Path("data/processed/reviews_IMDB_test.csv"),
    usecols=["review", "voted_up"],
    dtype={"review": "string", "voted_up": "int64"},
)
imdb_test_df = imdb_test_df.dropna(subset=["review", "voted_up"]).copy()
external_test_dfs["IMDB"] = imdb_test_df
print(f"IMDB: {len(imdb_test_df)} external test reviews")


WARDOGS: 1826 external test reviews
SQUAD: 1464 external test reviews
CYBERPUNK_2077: 1558 external test reviews
PERSONA_5_ROYAL: 1527 external test reviews
BALDURS_GATE_3: 1598 external test reviews
DOTA2: 967 external test reviews
HOLLOW_KNIGHT_SILKSONG: 1516 external test reviews
STARDEW_VALLEY: 1445 external test reviews
ELDEN_RING: 1476 external test reviews
FORZA_HORIZON_6: 1532 external test reviews
AMONG_US: 1301 external test reviews
IMDB: 4000 external test reviews


In [8]:
from sklearn.metrics import accuracy_score

# Reuse the fitted pipeline: predict transforms text without fitting TF-IDF again.
external_results = []
for game, external_df in external_test_dfs.items():
    y_true = external_df["voted_up"]
    y_pred = model.predict(external_df["review"])
    print(f"\n{game}")
    print(classification_report(
        y_true, y_pred, labels=[1, 0],
        target_names=(["positive", "negative"] if game == "IMDB"
                      else ["recommended", "not recommended"]),
        zero_division=0, digits=3,
    ))
    print("Confusion matrix (rows=true, columns=predicted; order=[1, 0]):")
    print(confusion_matrix(y_true, y_pred, labels=[1, 0]))
    external_results.append({
        "game": game,
        "samples": len(external_df),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    })

external_results = pd.DataFrame(external_results).set_index("game")
print("\nExternal test summary:")
print(external_results.to_string(float_format=lambda value: f"{value:.4f}"))



WARDOGS
                 precision    recall  f1-score   support

    recommended      0.944     0.576     0.716       883
not recommended      0.709     0.968     0.819       943

       accuracy                          0.779      1826
      macro avg      0.827     0.772     0.767      1826
   weighted avg      0.823     0.779     0.769      1826

Confusion matrix (rows=true, columns=predicted; order=[1, 0]):
[[509 374]
 [ 30 913]]

SQUAD
                 precision    recall  f1-score   support

    recommended      0.919     0.671     0.776       611
not recommended      0.803     0.958     0.873       853

       accuracy                          0.838      1464
      macro avg      0.861     0.814     0.825      1464
   weighted avg      0.851     0.838     0.833      1464

Confusion matrix (rows=true, columns=predicted; order=[1, 0]):
[[410 201]
 [ 36 817]]

CYBERPUNK_2077
                 precision    recall  f1-score   support

    recommended      0.887     0.779     0.830  